# Virtual Try-On (Colab)

Corre o CatVTON na GPU do Colab e lanca a app Gradio com link publico.

Antes de comecar: Runtime > Change runtime type > T4 GPU.

## 1. Confirmar GPU

In [ ]:
!nvidia-smi

## 2. Clonar o repositório do projeto + o código do CatVTON

In [ ]:
import os, sys

if not os.path.exists('/content/repo'):
    !git clone https://github.com/goncalvesdaniel99/2026-ei-aoopii-a20776 /content/repo
if not os.path.exists('/content/CatVTON'):
    !git clone https://github.com/Zheng-Chong/CatVTON /content/CatVTON

for p in ('/content/CatVTON', '/content/repo/src'):
    if p not in sys.path:
        sys.path.insert(0, p)

print('OK')

## 3. Instalar dependencias

Versoes fixas que o CatVTON precisa. Depois desta celula: Runtime > Restart session e correr de novo a celula 2.

In [ ]:
!pip install -q diffusers==0.29.2 transformers==4.27.3 accelerate==0.31.0     huggingface_hub==0.23.4 gradio==4.44.1 opencv-python-headless einops

## 4. Dados

Faz upload do data.zip (com a pasta test/ la dentro).

In [ ]:
from google.colab import files
import os, glob, shutil

up = files.upload()
zip_name = next(iter(up))
!rm -rf /content/_dataz && mkdir -p /content/_dataz
!unzip -q "{zip_name}" -d /content/_dataz

test_src = next(p for p in glob.glob('/content/_dataz/**/test', recursive=True)
                if os.path.isdir(os.path.join(p, 'image')))
os.makedirs('/content/repo/data', exist_ok=True)
dst = '/content/repo/data/test'
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.move(test_src, dst)

DATA_DIR = '/content/repo/data/test'
assert os.path.isdir(os.path.join(DATA_DIR, 'image')), 'Dados em falta'
print('Dados OK:', sorted(os.listdir(DATA_DIR)))

## 5. Teste rapido

Vestir a pessoa 0 com uma peca diferente para confirmar que o resultado e fiel. A primeira vez descarrega os pesos.

In [ ]:
from tryon import DatasetLoader, VirtualTryOn

data = DatasetLoader(DATA_DIR)
engine = VirtualTryOn(backend='catvton')

person_id = data.get_available_models()[0]
cloth_id  = data.get_available_garments()[5]
print('Pessoa:', person_id, '| Peca:', cloth_id)

result = engine.generate(
    data.load_person(person_id),
    data.load_cloth(cloth_id),
    data.load_mask(person_id),
)
result

## 6. Lancar a app

Gera um link publico. Deixa a celula a correr durante a demo.

In [ ]:
%env TRYON_BACKEND=catvton
%env GRADIO_SHARE=1
%env PYTHONPATH=/content/CatVTON
!cd /content/repo && python src/app.py

### Problemas comuns

- No module named 'model': re-corre a celula 2.
- OOM na T4: baixa width/height ou steps no CatVTONBackend.
- Sessao expira: reabre e corre as celulas de novo.